# Colab bootstrap — Plant Disease Classifier Reliability Study

Thin entry point. All logic lives in the repository (`src/plantrel`, `scripts/`); this notebook only
mounts Drive, fetches the code, records the environment, stages approved archives and points at
persistent storage.

**Status (Phase 2):** nothing here has been run on Colab yet. No dataset is downloaded or approved,
no model exists, and no credentials belong in this notebook. Edit the placeholders in the next cell first.

In [ ]:
# ---- 1. Placeholders: edit these. Nothing below runs against placeholder values. ----
REPO_URL   = "https://github.com/<OWNER>/<REPO>.git"   # PLACEHOLDER: the GitHub repository
REVISION   = "main"                                     # use a commit hash or tag for any real experiment
DRIVE_ROOT = "/content/drive/MyDrive/plant-disease-reliability"   # persistent artefacts (checkpoints, logs, archives)
LOCAL_ROOT = "/content/work"                            # fast local runtime disk (lost when the runtime ends)

# Approved dataset archives on Drive. EMPTY until Phase 3/4 approves a distribution.
# Format: {"name": {"drive_path": "<path under DRIVE_ROOT>", "sha256": "<64 hex>"}}
ARCHIVES = {}

assert "<OWNER>" not in REPO_URL, "Replace the REPO_URL placeholder before running."

In [ ]:
# ---- 2. Mount Google Drive (interactive prompt; no secrets stored) ----
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---- 3. Clone the repository at a fixed revision ----
# Public repository assumed. For a private repository use Colab "Secrets" (key icon) and read the token
# with google.colab.userdata at run time. Never paste a token into this notebook or commit one.
import os, subprocess
REPO_DIR = f"{LOCAL_ROOT}/repo"
os.makedirs(LOCAL_ROOT, exist_ok=True)
if not os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", REVISION], check=True)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
dirty = subprocess.run(["git", "-C", REPO_DIR, "status", "--porcelain"], capture_output=True, text=True).stdout.strip()
print("Git revision:", head, "| working tree dirty:", bool(dirty))

In [ ]:
# ---- 4. Install the package WITHOUT touching Colab's pre-installed PyTorch ----
# --no-deps keeps the existing torch/torchvision/numpy builds. If an import fails, install only the
# specific missing package; do not downgrade or reinstall torch.
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", REPO_DIR], check=True)
import importlib
for mod in ["plantrel", "numpy", "scipy", "pandas", "yaml", "PIL", "torch", "torchvision"]:
    try:
        importlib.import_module(mod); print("import ok  :", mod)
    except Exception as exc:
        print("import FAIL:", mod, type(exc).__name__, exc)

In [ ]:
# ---- 5. Record the environment and check the GPU (every session) ----
import datetime, json, torch
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)
stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
env_log = f"{DRIVE_ROOT}/logs/env_{stamp}.json"
subprocess.run(["python", f"{REPO_DIR}/scripts/log_environment.py", "--out", env_log], check=True)
print("Environment record:", env_log)
if not torch.cuda.is_available():
    print("NO GPU assigned. Do not start training in this session. Evaluation from saved logits can run on CPU.")
else:
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
    print("(An earlier session reported a Tesla T4; a new session may differ. Trust this output, not that note.)")

In [ ]:
# ---- 6. Stage approved archives: copy Drive -> local disk, verify SHA-256, extract locally ----
staged = {}
if not ARCHIVES:
    print("No approved archives configured yet (expected in Phase 2). Nothing copied.")
for name, spec in ARCHIVES.items():
    out = subprocess.run(
        ["python", f"{REPO_DIR}/scripts/stage_archive.py",
         "--src", f"{DRIVE_ROOT}/{spec['drive_path']}", "--local-dir", f"{LOCAL_ROOT}/archives",
         "--sha256", spec["sha256"], "--extract-to", f"{LOCAL_ROOT}/data/{name}"],
        capture_output=True, text=True)
    print(name, "->", "OK" if out.returncode == 0 else "FAILED", out.stderr.strip() or "")
    if out.returncode == 0:
        staged[name] = json.loads(out.stdout)

In [ ]:
# ---- 7. Persistent locations for checkpoints and outputs (Drive survives the runtime) ----
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR  = f"{DRIVE_ROOT}/outputs"
for d in (CKPT_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Checkpoints:", CKPT_DIR, "\nOutputs    :", OUT_DIR)

## Next

Later phases call repository scripts from here, for example `python scripts/train.py ...`, which
refuses to run without a verified dataset freeze. Do not copy implementation code into notebook cells.
Streamlit demos cannot be served from free Colab runtimes; the demo is planned for AWS in a later phase.